# Slide deck datasets and assets

This notebook generates reproducible Excel sources and PNG assets for the Quarto slide deck.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path('../../')
raw_path = ROOT / 'data/raw/cars.xlsx'
iv_path = ROOT / 'data/bivariate_info/win_loss_iv_report.xlsx'
classification_path = ROOT / 'slidedeck/data/classification_model_report.xlsx'
regression_path = ROOT / 'slidedeck/data/regression_model_report.xlsx'
slide_data_dir = ROOT / 'slidedeck/data'
asset_dir = ROOT / 'slidedeck/assets'
slide_data_dir.mkdir(parents=True, exist_ok=True)
asset_dir.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style='whitegrid')
sns.set_palette(['#2563EB', '#0F172A', '#14B8A6', '#F59E0B', '#DC2626'])
size_order = ['100K or less', '100K to 250K', '250K to 500K', '500K to 1M', 'More than 1M']

df = pd.read_excel(raw_path)
df['won_flag'] = (df['Opportunity Result'] == 'Won').astype(int)
df['funnel_phase'] = np.where(df['Revenue From Client Past Two Years (USD)'].astype(str).eq('0 (No business)'), '(Re)Acquisition', 'Engagement / upselling')
print('rows:', len(df))


rows: 78025


In [2]:
client_segmentation = (
    df.groupby('Client Size By Revenue (USD)', dropna=False)
      .agg(opportunities=('Opportunity Number', 'count'),
           win_rate=('won_flag', 'mean'),
           median_amount=('Opportunity Amount USD', 'median'))
      .reset_index()
      .rename(columns={'Client Size By Revenue (USD)': 'client_size'})
)
client_segmentation['client_size'] = pd.Categorical(client_segmentation['client_size'], categories=size_order, ordered=True)
client_segmentation = client_segmentation.sort_values('client_size').reset_index(drop=True)

two_phase_funnel = (
    df.groupby('funnel_phase', dropna=False)
      .agg(opportunities=('Opportunity Number', 'count'),
           win_rate=('won_flag', 'mean'),
           avg_amount=('Opportunity Amount USD', 'mean'))
      .reset_index()
)

funnel_x_segment = pd.pivot_table(
    df,
    index='funnel_phase',
    columns='Client Size By Revenue (USD)',
    values='won_flag',
    aggfunc='mean'
).reindex(columns=size_order)

other_variables = pd.DataFrame({
    'metric': ['total_days_p50', 'total_days_p90', 'amount_p50', 'amount_p90'],
    'value': [
        df['Total Days Identified Through Qualified'].median(),
        df['Total Days Identified Through Qualified'].quantile(0.90),
        df['Opportunity Amount USD'].median(),
        df['Opportunity Amount USD'].quantile(0.90),
    ]
})

iv_top = pd.read_excel(iv_path, sheet_name='variable_summaries').head(10).copy()

eda_excel_path = slide_data_dir / 'eda_slide_data.xlsx'
with pd.ExcelWriter(eda_excel_path, engine='openpyxl') as writer:
    client_segmentation.to_excel(writer, sheet_name='client_segmentation', index=False)
    two_phase_funnel.to_excel(writer, sheet_name='two_phase_funnel', index=False)
    funnel_x_segment.reset_index().to_excel(writer, sheet_name='funnel_x_segment', index=False)
    other_variables.to_excel(writer, sheet_name='other_variables', index=False)
    iv_top.to_excel(writer, sheet_name='iv_top_variables', index=False)

print('saved:', eda_excel_path)


saved: ../../slidedeck/data/eda_slide_data.xlsx


In [3]:
classification_summary = pd.read_excel(classification_path, sheet_name='cv_summary')
classification_cv_selected = pd.read_excel(classification_path, sheet_name='cv_selected_model')
classification_test_metrics = pd.read_excel(classification_path, sheet_name='test_metrics')
classification_importance = pd.read_excel(classification_path, sheet_name='feature_importance').head(10)
classification_lift = pd.read_excel(classification_path, sheet_name='prioritization_lift')
classification_metadata = pd.read_excel(classification_path, sheet_name='metadata')

regression_summary = pd.read_excel(regression_path, sheet_name='cv_summary')
regression_cv_selected = pd.read_excel(regression_path, sheet_name='cv_selected_model')
regression_test_metrics = pd.read_excel(regression_path, sheet_name='test_metrics')
regression_importance = pd.read_excel(regression_path, sheet_name='feature_importance').head(10)
regression_forecast = pd.read_excel(regression_path, sheet_name='forecast_summary')
regression_predictions = pd.read_excel(regression_path, sheet_name='test_predictions')
regression_metadata = pd.read_excel(regression_path, sheet_name='metadata')

selected_models = pd.DataFrame([
    {
        'model': 'Dynamic win/loss classification',
        'target': 'Opportunity Result',
        'business_use': 'Prioritization and pipeline ranking',
        'data_used': 'All available snapshot variables, including process timing fields',
        'notes': f"Selected logistic model: {classification_metadata.loc[0, 'selected_experiment']}",
    },
    {
        'model': 'Dynamic amount regression',
        'target': 'Opportunity Amount USD',
        'business_use': 'Sizing and aggregate forecasting',
        'data_used': 'All available snapshot variables, including process timing fields',
        'notes': f"Selected regression model: {regression_metadata.loc[0, 'selected_experiment']}",
    },
])

model_excel_path = slide_data_dir / 'model_slide_data.xlsx'
with pd.ExcelWriter(model_excel_path, engine='openpyxl') as writer:
    selected_models.to_excel(writer, sheet_name='selected_models', index=False)
    classification_summary.to_excel(writer, sheet_name='classification_summary', index=False)
    classification_cv_selected.to_excel(writer, sheet_name='classification_cv_selected', index=False)
    classification_test_metrics.to_excel(writer, sheet_name='classification_test_metrics', index=False)
    classification_importance.to_excel(writer, sheet_name='classification_importance', index=False)
    classification_lift.to_excel(writer, sheet_name='prioritization_lift', index=False)
    classification_metadata.to_excel(writer, sheet_name='classification_metadata', index=False)
    regression_summary.to_excel(writer, sheet_name='regression_summary', index=False)
    regression_cv_selected.to_excel(writer, sheet_name='regression_cv_selected', index=False)
    regression_test_metrics.to_excel(writer, sheet_name='regression_test_metrics', index=False)
    regression_importance.to_excel(writer, sheet_name='regression_importance', index=False)
    regression_forecast.to_excel(writer, sheet_name='forecast_summary', index=False)
    regression_predictions.head(5000).to_excel(writer, sheet_name='forecast_predictions_sample', index=False)
    regression_metadata.to_excel(writer, sheet_name='regression_metadata', index=False)

metric_tables_path = slide_data_dir / 'model_metric_tables.xlsx'
with pd.ExcelWriter(metric_tables_path, engine='openpyxl') as writer:
    classification_cv_selected.to_excel(writer, sheet_name='classification_cv', index=False)
    classification_test_metrics.to_excel(writer, sheet_name='classification_test', index=False)
    regression_cv_selected.to_excel(writer, sheet_name='regression_cv', index=False)
    regression_test_metrics.to_excel(writer, sheet_name='regression_test', index=False)

print('saved:', model_excel_path)
print('saved:', metric_tables_path)


saved: ../../slidedeck/data/model_slide_data.xlsx
saved: ../../slidedeck/data/model_metric_tables.xlsx


In [4]:
fig, ax1 = plt.subplots(figsize=(10, 5))
sns.barplot(data=client_segmentation, x='client_size', y='opportunities', color='#93C5FD', ax=ax1)
ax1.set_xlabel('Client size by revenue')
ax1.set_ylabel('Opportunity count', color='#1D4ED8')
ax1.tick_params(axis='x', rotation=20)
ax2 = ax1.twinx()
sns.lineplot(data=client_segmentation, x='client_size', y='win_rate', marker='o', linewidth=2.5, color='#0F172A', ax=ax2)
ax2.set_ylabel('Observed win rate', color='#0F172A')
ax2.set_ylim(0, max(0.35, client_segmentation['win_rate'].max() * 1.15))
ax2.grid(False)
fig.tight_layout()
fig.savefig(asset_dir / 'eda_client_segmentation.png', dpi=220, bbox_inches='tight')
plt.close(fig)

fig, ax1 = plt.subplots(figsize=(8, 5))
sns.barplot(data=two_phase_funnel, x='funnel_phase', y='opportunities', color='#60A5FA', ax=ax1)
ax1.set_xlabel('Funnel phase')
ax1.set_ylabel('Opportunity count', color='#1D4ED8')
ax2 = ax1.twinx()
sns.lineplot(data=two_phase_funnel, x='funnel_phase', y='win_rate', marker='o', linewidth=2.5, color='#0F172A', ax=ax2)
ax2.set_ylabel('Observed win rate', color='#0F172A')
ax2.set_ylim(0, max(0.4, two_phase_funnel['win_rate'].max() * 1.2))
ax2.grid(False)
fig.tight_layout()
fig.savefig(asset_dir / 'eda_two_phase_funnel.png', dpi=220, bbox_inches='tight')
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.heatmap(funnel_x_segment, annot=True, fmt='.1%', cmap='Blues', linewidths=0.5, cbar_kws={'label': 'Observed win rate'}, ax=ax)
ax.set_xlabel('Client size by revenue')
ax.set_ylabel('Funnel phase')
fig.tight_layout()
fig.savefig(asset_dir / 'eda_funnel_x_segment.png', dpi=220, bbox_inches='tight')
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 5))
iv_plot = iv_top.sort_values('iv', ascending=True)
sns.barplot(data=iv_plot, x='iv', y='variable', color='#2563EB', ax=ax)
ax.set_xlabel('Information value')
ax.set_ylabel('Variable')
fig.tight_layout()
fig.savefig(asset_dir / 'eda_iv_ranking.png', dpi=220, bbox_inches='tight')
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.histplot(df['Total Days Identified Through Qualified'].dropna(), bins=30, color='#2563EB', ax=axes[0])
axes[0].set_title('Total days in qualifying')
axes[0].set_xlabel('Days')
amount_cap = df['Opportunity Amount USD'].quantile(0.95)
sns.histplot(df.loc[df['Opportunity Amount USD'] <= amount_cap, 'Opportunity Amount USD'].dropna(), bins=30, color='#14B8A6', ax=axes[1])
axes[1].set_title('Opportunity amount (<= 95th percentile)')
axes[1].set_xlabel('Amount USD')
fig.tight_layout()
fig.savefig(asset_dir / 'eda_other_vars.png', dpi=220, bbox_inches='tight')
plt.close(fig)

print('EDA assets regenerated')


EDA assets regenerated


In [5]:
fig, ax = plt.subplots(figsize=(7.5, 5))
class_plot = classification_importance.sort_values('importance_mean', ascending=True)
sns.barplot(data=class_plot, x='importance_mean', y='feature', color='#2563EB', ax=ax)
ax.set_title('Classification model importance')
ax.set_xlabel('Sum of absolute coefficient magnitudes across transformed terms')
ax.set_ylabel('Feature')
fig.tight_layout()
fig.savefig(asset_dir / 'classification_importance.png', dpi=220, bbox_inches='tight')
plt.close(fig)

fig, ax = plt.subplots(figsize=(7.5, 5))
reg_plot = regression_importance.copy()
reg_plot['display_feature'] = (reg_plot['feature'].astype(str)
    .str.replace('categorical__', '', regex=False)
    .str.replace('numerical__', '', regex=False)
    .str.replace('_', ' ', regex=False))
reg_plot = reg_plot.sort_values('importance_mean', ascending=True)
sns.barplot(data=reg_plot, x='importance_mean', y='display_feature', color='#14B8A6', ax=ax)
ax.set_title('Regression model importance')
ax.set_xlabel('Absolute coefficient magnitude')
ax.set_ylabel('Feature')
fig.tight_layout()
fig.savefig(asset_dir / 'regression_importance.png', dpi=220, bbox_inches='tight')
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 5))
plot_lift = classification_lift.sort_values('score_decile')
sns.barplot(data=plot_lift, x='score_decile', y='share_of_total_wins', color='#2563EB', ax=ax)
ax.plot(range(len(plot_lift)), [1/len(plot_lift)] * len(plot_lift), color='#0F172A', linestyle='--', label='Uniform baseline')
ax.set_xlabel('Propensity score decile (1 = lowest, 10 = highest)')
ax.set_ylabel('Share of total wins')
ax.legend()
fig.tight_layout()
fig.savefig(asset_dir / 'sim_prioritization.png', dpi=220, bbox_inches='tight')
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 5))
plot_df = regression_predictions.sample(min(len(regression_predictions), 2000), random_state=42)
sns.scatterplot(data=plot_df, x='actual_amount', y='predicted_amount', alpha=0.5, s=30, color='#2563EB', ax=ax)
max_val = float(max(plot_df['actual_amount'].max(), plot_df['predicted_amount'].max()))
ax.plot([0, max_val], [0, max_val], linestyle='--', color='#0F172A', label='Perfect forecast')
ax.set_xlabel('Actual amount')
ax.set_ylabel('Predicted amount')
ax.legend()
fig.tight_layout()
fig.savefig(asset_dir / 'sim_forecasting.png', dpi=220, bbox_inches='tight')
plt.close(fig)

print('model and simulation assets regenerated')


model and simulation assets regenerated
